## Initialization

### Imports

In [ ]:
# Importing needed code

import re
import json
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    TypeVar,
    Union,
    # Any,
    Literal
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log, ceil
import shutil
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
from mpl_toolkits.axes_grid1 import make_axes_locatable
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from scipy.interpolate import make_interp_spline
from scipy.stats import linregress
from scipy.signal import savgol_filter, find_peaks, peak_prominences, peak_widths
from pint import Quantity
import bottleneck

from data_processing import processing as proc
from data_processing import loading as load
from data_processing import types as proc_types
from data_processing import helpers
from data_processing.paths import (
    get_report_root, get_exp_root, get_reactor_data_root)
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    # NonReactorDataframeColumn,
    # SliceFitDataframeColumn,
    EnergyColumn,
    get_df_col
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing.processing.neutron_window_strategy.strategy_factory \
    import NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy \
    import AbstractNeutronStrategy
# from data_processing.helpers import (
#     # get_input_with_default,
#     # get_input_required,
#     # input_experiment_ids,
#     stop,
#     get_midpoints_from_bins
# )


### Functions

In [ ]:
def energy_resolution_fn(E, alpha, beta, gamma):
    return np.sqrt(np.square(alpha) + np.square(beta) / E + np.square(gamma / E))

## Plotting

### Plot Style Constants

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"
bg_green = "#21d894"
transparent = "#00000000"

### Plot Functions

In [ ]:
def plot_figure_19(
    ax: mpl.axes.Axes
):
    alpha = 0.102
    beta = 0.102
    gamma = 0.036
    plot_x = np.linspace(1e-6, 3, 150)
    plot_y = energy_resolution_fn(plot_x, alpha, beta, gamma)

    ax.plot(
        plot_x, plot_y, "-",
        lw=3,
        color=bg_blue,
    )
    ax.set_xlim(0, 3)
    ax.set_ylim(0, 0.4)
    ax.set_xlabel(r"Light output ($E$, MeVee)", fontsize=fontsize)
    ax.set_ylabel(r"Energy resolution ($\frac{{\Delta}E}{E}$)", fontsize=fontsize)
    ax.tick_params(labelsize=fontsize)
    ax.grid()

### Figure Creation

In [ ]:
fig, ax = plt.subplots(layout="constrained")
plot_figure_19(ax)